# OMNI-906 · The chosen PFM on held-out data: SM-2, SM-3, false alarms, TC-5 and the scene control (Colab GPU)

Runs the repo's own code from a clone of the branch: `scripts/pfm_heldout.py` (the measures, D-60),
`scripts/train_pfm.py` (loading and predicting), `scripts/pfm_samples.py` (the committed samples) and
`scripts/pfm_runs.py` (the chosen run, checked by SHA-256). The rules were written before this run (D-60):

- **The model and `θ_d`** are OMNI-905's chosen run (`model_runs.json`), unchanged: nothing is fit on held-out
- **SM-2:** a positive window's class in the top 3 at its last grid point. CCD has one event class, so top-3 is
  always met there; top-1 and ROC-AUC beside it
- **SM-3:** on CCD's held-out crash clips, the time to the accident at the first warning; target ≥ 1.80 s (D-23)
- **False alarms:** the share of held-out negative windows with any warning
- **TC-5, first half:** one row per window (UCF-Crime 30 s and 60 s, CCD) → `results/omni906_windows.csv`
- **The scene control (K-35, K-36):** the same model given only each window's first grid vector (on a
  `minus_first` model, D-61, this input is empty by construction: the evidence there is the timeline and the rise)
- **The rise (D-61):** mean `p(event)` in the last 10 s before the event minus 20-30 s before it

1. **Setup:** clone, Drive
2. **The held-out vectors and the model:** both domains' held-out InternVideo2 vectors copied from Drive; the
   chosen run copied and checked
3. **The measures:** model and control, per domain (and per length on UCF-Crime) →
   `results/omni906_heldout.json`, `results/omni906_windows.csv`
4. **The figure:** `p(event)` against the time to the event, model and control; CCD's lead times →
   `results/omni906_timeline.png`

Cell 2's `RUN_ID`: empty for the chosen run, or a run id (OMNI-914's `pfm-joint-v2-mf-...`). The files are named
`omni906_*` for OMNI-905's run and `heldout_{run_id}_*` for any other.

Runtime → Change runtime type → **GPU** (a T4 is enough: evaluation only, a few minutes).

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, "/content/omni-sight/scripts")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
print(torch.__version__, GPU)

In [ ]:
# 2 · The held-out vectors (copied from Drive once) and the chosen model
import csv, hashlib, json, os, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pfm_heldout, pfm_runs, pfm_samples, train_pfm

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
LOCAL = Path("/content/pfm")
DOMAINS = ("ccd", "ucf_crime")
STEP = {"ccd": 0.5, "ucf_crime": 5.0}  # the timeline's bins of time to the event
SAMPLE_FILES = {d: REPO_DIR / "datasets" / "samples" / f"{d}.csv" for d in DOMAINS}
SAMPLES = {d: [s for s in pfm_samples.read_samples(SAMPLE_FILES[d]) if s["split"] == "held_out"] for d in DOMAINS}

def copy_folder(domain):
    keys = sorted({s["clip_key"] for s in SAMPLES[domain]})
    source, target = DRIVE / "embeddings" / "pfm" / domain / "internvideo2", LOCAL / domain
    def one(key):
        dst = pfm_samples.out_file(target, key)
        if not dst.is_file():  # copied under a temporary name, renamed when complete: a cut copy leaves no broken file
            dst.parent.mkdir(parents=True, exist_ok=True)
            part = dst.with_name(dst.name + ".part")
            shutil.copyfile(pfm_samples.out_file(source, key), part)
            os.replace(part, dst)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(one, keys))
    return target

HELD = {}
for d in DOMAINS:
    start = time.time()
    store = pfm_samples.Store(SAMPLES[d], copy_folder(d))
    HELD[d] = train_pfm.Data(store, SAMPLES[d], range(len(SAMPLES[d])), d)
    print(f"{d}: held-out {len(HELD[d]):,} samples, {len(set(HELD[d].window)):,} windows ({time.time() - start:.0f} s)")

RUN_ID = ""  # @param {type:"string"}
# empty: the chosen run in model_runs.json; or a run id (e.g. OMNI-914's), SHA-256 checked
RUN = pfm_runs.download(DRIVE, RUN_ID or None, target=LOCAL / "model")
NAME = "omni906" if RUN["run_id"] == "pfm-joint-v1-s1-e8" else f"heldout_{RUN['run_id']}"  # the output files' names
MODEL, STATE = train_pfm.load(LOCAL / "model" / pfm_runs.CHECKPOINT, DEVICE)
THETA = RUN["thresholds"]
print(RUN["run_id"], "epoch", STATE["epoch"], "θ", THETA)

In [ ]:
# 3 · The measures: the model and the scene control, per domain
RESULTS = {}
def show(name, s):
    lead = f"{s['lead_mean_s']:.2f} s (quartiles {', '.join(f'{q:.1f}' for q in s['lead_quartiles_s'])})" if s["lead_mean_s"] is not None else "-"
    print(f"  {name:16s} AUC {s['roc_auc']:.4f}  top-1 {s['top1']:.4f}  top-3 windows {s['top3_windows']:.3f} "
          f"(top-1 {s['top1_windows']:.3f})  Δt MAE {s['dt_mae_s']:.2f} s (mean {s['dt_mean_baseline_s']:.2f})  "
          f"false alarms {s['false_alarm_windows']:.3f} of {s['negative_windows']}  warned {s['warned_windows']}/{s['positive_windows']}  "
          f"lead {lead}, ceiling {s['lead_ceiling_s'] or 0:.2f} s; true class first {s['warned_correct_windows']}; "
          f"rise {s['rise']:+.3f}")

for d in DOMAINS:
    RESULTS[d] = {}
    for name, data in (("model", HELD[d]), ("control", pfm_heldout.control(HELD[d]))):
        p, dt = train_pfm.predict(MODEL, data, DEVICE)
        RESULTS[d][name] = pfm_heldout.measure(data, p, dt, THETA[d], STEP[d])
    print(d)
    for name in ("model", "control"):
        show(name, RESULTS[d][name]["summary"])
        for length, s in RESULTS[d][name].get("per_length", {}).items():
            show(f"{name} {length} s", s)

print("\nper class (model; control's p(event) at the first point in brackets)")
for d in DOMAINS:
    for label, c in RESULTS[d]["model"]["by_class"].items():
        k = RESULTS[d]["control"]["by_class"][label]
        lead = "-" if c.get("lead_mean_s") is None else f"{c['lead_mean_s']:.1f}"
        extra = (f"top-3 {c['top3_last']:.2f}  lead {lead} / {c['ceiling_s']:.1f} s"
                 if label != "Normal" else "")
        print(f"  {d:9s} {label:13s} {c['windows']:3d} windows  warned {c['warned']:.2f}  p(event) first {c['p_event_first']:.2f} "
              f"last {c['p_event_last']:.2f} [{k['p_event_first']:.2f}]  {extra}")

OUTDIR = DRIVE / "results"
OUTDIR.mkdir(parents=True, exist_ok=True)
COLUMNS = ["domain", "input", "clip_key", "length_s", "label", "samples", "warned", "top3_last", "top1_last", "lead_s",
           "lead_correct_s", "first_warning", "ceiling_s", "p_event_first", "p_event_last"]
with open(OUTDIR / f"{NAME}_windows.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNS, lineterminator="\n")
    writer.writeheader()
    for d in DOMAINS:
        for name in ("model", "control"):
            for row in RESULTS[d][name]["windows"]:
                writer.writerow({"domain": d, "input": name, **row})
(OUTDIR / f"{NAME}_heldout.json").write_text(json.dumps({
    "commit": COMMIT, "gpu": GPU, "torch": torch.__version__, "run_id": RUN["run_id"], "thresholds": THETA,
    "checkpoint_sha256": RUN["checkpoint_sha256"],
    "samples_sha256": {d: hashlib.sha256(SAMPLE_FILES[d].read_bytes()).hexdigest() for d in DOMAINS},
    "results": {d: {n: {k: v for k, v in r.items() if k != "windows"} for n, r in RESULTS[d].items()} for d in DOMAINS},
}, indent=1, default=float))
print(OUTDIR / f"{NAME}_heldout.json", OUTDIR / f"{NAME}_windows.csv")

In [ ]:
# 4 · The figure: p(event) as the event nears, model and control; CCD's lead times
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, d in zip(axes, DOMAINS):
    for name, style in (("model", "-o"), ("control", "--s")):
        tl = RESULTS[d][name]["timeline"]
        x = np.array(tl["tau_bin_start_s"]) + STEP[d] / 2
        ax.plot(x, tl["positive_mean"], style, ms=3, label=f"{name}: lead-up windows")
        ax.axhline(tl["negative_mean"], ls=":", color=ax.lines[-1].get_color(), label=f"{name}: normal windows")
    ax.invert_xaxis()
    ax.set(title=f"{d}: mean p(event) before the event", xlabel="time to the event (s)", ylabel="1 - p(Normal)", ylim=(0, 1))
    ax.legend(fontsize=8)
s = RESULTS["ccd"]["model"]["summary"]
h = s.get("lead_histogram")
if h:
    axes[2].bar(h["edges"][:-1], h["counts"], width=np.diff(h["edges"]), align="edge", edgecolor="white")
    axes[2].axvline(s["lead_mean_s"], color="black", label=f"mean {s['lead_mean_s']:.2f} s")
axes[2].axvline(1.80, color="red", ls="--", label="SM-3 target 1.80 s")
axes[2].set(title=f"CCD: lead time ({s['warned_windows']}/{s['positive_windows']} clips warned)",
            xlabel="time to the accident at the first warning (s)", ylabel="clips")
axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTDIR / f"{NAME}_timeline.png", dpi=150)
print(OUTDIR / f"{NAME}_timeline.png")